In [14]:
import os

# set your default path here
os.chdir("/Users/aditya/Downloads/Medisyn_Labs_case_study")

# confirm
print("Current Working Directory:", os.getcwd())


Current Working Directory: /Users/aditya/Downloads/Medisyn_Labs_case_study


In [17]:
# ==============================================================================
# Step 1: Setup and Data Loading
# ==============================================================================
import pandas as pd

# Load and combine the datasets
try:
    train_df = pd.read_csv('train.tsv', sep='\t', header=None)
    test_df = pd.read_csv('test.tsv', sep='\t', header=None)
except FileNotFoundError:
    print("Ensure 'train.tsv' and 'test.tsv' are in the same directory.")
    exit()

# Define and assign column names
column_names = [
    'CustomerID', 'Medicine', 'Rating', 'Effectiveness', 'Side Effects',
    'Condition', 'Benefits', 'Side Effects Review', 'Comments'
]
train_df.columns = column_names
test_df.columns = column_names
df = pd.concat([train_df, test_df], ignore_index=True)

# ==============================================================================
# Step 2: Feature Engineering for Ranking
# ==============================================================================
effectiveness_map = {
    'Ineffective': 1,
    'Marginally Effective': 2,
    'Moderately Effective': 3,
    'Considerably Effective': 4,
    'Highly Effective': 5
}
df['Effectiveness_Num'] = df['Effectiveness'].map(effectiveness_map)
df.dropna(subset=['Condition', 'Medicine', 'Rating', 'Effectiveness_Num'], inplace=True)

print("Data prepared for the recommendation engine.")

# ==============================================================================
# Step 3: Create the Recommendation Function
# ==============================================================================
def get_medicine_recommendations(symptom, top_n=5):
    """
    Recommends the best medicines for a given symptom based on past reviews.
    """
    print(f"Searching for top {top_n} medicine recommendations for: '{symptom}'\n")

    condition_df = df[df['Condition'].str.contains(symptom, case=False)]
    if condition_df.empty:
        return f"Sorry, we could not find any reviews for the symptom '{symptom}'."

    medicine_summary = condition_df.groupby('Medicine').agg(
        review_count=('CustomerID', 'count'),
        avg_rating=('Rating', 'mean'),
        avg_effectiveness=('Effectiveness_Num', 'mean')
    ).reset_index()

 
    reliable_medicines = medicine_summary[medicine_summary['review_count'] >= 3].copy()

    if reliable_medicines.empty:
        return f"Sorry, we don't have enough reviews to confidently recommend a medicine for '{symptom}'."

    reliable_medicines['score'] = (reliable_medicines['avg_rating'] * 0.7) + \
                                  (reliable_medicines['avg_effectiveness'] * 0.3)

    top_recommendations = reliable_medicines.sort_values(by='score', ascending=False).head(top_n)
    return top_recommendations[['Medicine', 'review_count', 'avg_rating', 'score']].round(2)

# ==============================================================================
# Step 4: Use the Recommender (Example Usage)
# ==============================================================================
pain_recommendations = get_medicine_recommendations('pain')
print(pain_recommendations)

Data prepared for the recommendation engine.
Searching for top 5 medicine recommendations for: 'pain'

         Medicine  review_count  avg_rating  score
54         valium             3        9.33   8.03
32  nortriptyline             3        7.67   6.67
21         lortab             4        7.00   6.25
35      oxycodone             9        6.89   6.16
5        celebrex             8        7.00   6.06
